# 🏆 FIFA World Cup 2026 — Data Collection
**Notebook 01 of 04**

This notebook collects and saves all raw data needed for the prediction project:

| # | Dataset | Source | Rows |
|---|---------|--------|------|
| 1 | Historical match results (1872–2026) | GitHub (martj42) | ~49,400 |
| 2 | Goalscorers history | GitHub (martj42) | ~47,600 |
| 3 | Penalty shootouts | GitHub (martj42) | ~677 |
| 4 | WC team history w/ rankings (1994–2022) | GitHub (zvizdo) | ~248 |
| 5 | WC 2026 — 48 teams, groups, FIFA ranks | GitHub (zvizdo) | 48 |
| 6 | WC 2026 — Full fixture list (104 matches) | Constructed from official draw | 104 |
| 7 | Current FIFA rankings (April 2026) | Hardcoded from FIFA.com | 48 |


## 0. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import requests
import json
import os
from pathlib import Path
from io import StringIO

# ── Project paths ────────────────────────────────────────────────
BASE_DIR   = Path("")          # worldcup2026/
RAW_DIR    = BASE_DIR / "data" / "raw"
PROC_DIR   = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

print(" Directories ready")
print(f"   Raw data  → {RAW_DIR.resolve()}")
print(f"   Processed → {PROC_DIR.resolve()}")

 Directories ready
   Raw data  → G:\PYTHON\2025_f1_predictions-main\football\WC\data\raw
   Processed → G:\PYTHON\2025_f1_predictions-main\football\WC\data\processed


---
## 1. Historical International Match Results (1872–2026)

**Source:** [martj42/international_results](https://github.com/martj42/international_results) — the gold standard dataset for international football, actively maintained.

**Columns:** `date, home_team, away_team, home_score, away_score, tournament, city, country, neutral`

In [2]:
MARTJ42_BASE = "https://raw.githubusercontent.com/martj42/international_results/master"

def download_csv(url: str, save_path: Path, name: str) -> pd.DataFrame:
    """Download a CSV from a URL, save it, and return as DataFrame."""
    print(f"  Downloading {name}...")
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    df = pd.read_csv(StringIO(response.text))
    df.to_csv(save_path, index=False)
    print(f"    {name}: {len(df):,} rows  →  saved to {save_path.name}")
    return df

# Download all three datasets from martj42
df_results     = download_csv(f"{MARTJ42_BASE}/results.csv",     RAW_DIR / "results.csv",     "Match Results")
df_goals       = download_csv(f"{MARTJ42_BASE}/goalscorers.csv", RAW_DIR / "goalscorers.csv", "Goalscorers")
df_shootouts   = download_csv(f"{MARTJ42_BASE}/shootouts.csv",   RAW_DIR / "shootouts.csv",   "Shootouts")

    Match Results: 49,477 rows  →  saved to results.csv
    Goalscorers: 47,601 rows  →  saved to goalscorers.csv
    Shootouts: 678 rows  →  saved to shootouts.csv


In [3]:
# Quick look at the results dataset
print("=" * 55)
print("MATCH RESULTS — first 5 rows")
print("=" * 55)
display(df_results.head())

print(f"\n Date range: {df_results['date'].min()}  →  {df_results['date'].max()}")
print(f"  Total matches : {len(df_results):,}")
print(f" Unique teams  : {pd.unique(df_results[['home_team','away_team']].values.ravel()).shape[0]}")
print(f" Tournaments   : {df_results['tournament'].nunique()}")
print("\nTop 10 tournaments by match count:")
print(df_results['tournament'].value_counts().head(10).to_string())

MATCH RESULTS — first 5 rows


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False



 Date range: 1872-11-30  →  2026-06-27
  Total matches : 49,477
 Unique teams  : 336
 Tournaments   : 200

Top 10 tournaments by match count:
tournament
Friendly                                18388
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1036
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620


In [4]:
# Check the goalscorers and shootouts data
print("GOALSCORERS — columns:", df_goals.columns.tolist())
display(df_goals.head(3))

print("\nSHOOTOUTS — columns:", df_shootouts.columns.tolist())
display(df_shootouts.head(3))

GOALSCORERS — columns: ['date', 'home_team', 'away_team', 'team', 'scorer', 'minute', 'own_goal', 'penalty']


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
0,1916-07-02,Chile,Uruguay,Uruguay,José Piendibene,44.0,False,False
1,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,55.0,False,False
2,1916-07-02,Chile,Uruguay,Uruguay,Isabelino Gradín,70.0,False,False



SHOOTOUTS — columns: ['date', 'home_team', 'away_team', 'winner', 'first_shooter']


,date,home_team,away_team,winner,first_shooter
0,1967-08-22,India,Taiwan,Taiwan,NaN
1,1971-11-14,South Korea,Vietnam Republic,South Korea,NaN
2,1972-05-07,South Korea,Iraq,Iraq,NaN


---
## 2. World Cup Editions — Team Rankings History (1994–2022)

**Source:** Curated WC simulation project. Contains each team's FIFA rank at the time of their WC appearance. Key for training features.


In [5]:
WC_TEAMS_URL = "https://raw.githubusercontent.com/zvizdo/fifa-wc-2026-simulation/main/data/wc_teams.csv"

df_wc_teams = download_csv(WC_TEAMS_URL, RAW_DIR / "wc_teams_history.csv", "WC Teams History")
display(df_wc_teams.head(10))
print(f"\nWC editions covered: {sorted(df_wc_teams['year'].unique())}")

    WC Teams History: 248 rows  →  saved to wc_teams_history.csv


,year,team,confederation,rank
0,1994,Saudi Arabia,AFC,34
1,1994,South Korea,AFC,37
2,1994,Cameroon,CAF,24
3,1994,Morocco,CAF,28
4,1994,Nigeria,CAF,11
5,1994,Mexico,CONCACAF,16
6,1994,United States,CONCACAF,23
7,1994,Argentina,CONMEBOL,8
8,1994,Bolivia,CONMEBOL,43
9,1994,Brazil,CONMEBOL,3



WC editions covered: [np.int64(1994), np.int64(1998), np.int64(2002), np.int64(2006), np.int64(2010), np.int64(2014), np.int64(2018), np.int64(2022)]


---
## 3. WC 2026 — 48 Qualified Teams + Groups

**Source:** Curated from the official FIFA draw (December 5, 2025, Kennedy Center, Washington DC).

All 12 groups of 4 confirmed — includes FIFA rank at time of draw.


In [6]:
WC2026_TEAMS_URL = "https://raw.githubusercontent.com/zvizdo/fifa-wc-2026-simulation/main/data/wc_2026_teams.json"

print("  Downloading WC 2026 teams JSON...")
resp = requests.get(WC2026_TEAMS_URL, timeout=30)
resp.raise_for_status()
wc2026_raw = resp.json()

# Flatten the JSON into a DataFrame
rows = []
for group, teams in wc2026_raw["groups"].items():
    for team in teams:
        rows.append({
            "group": group,
            "team": team["name"],
            "confederation": team["confederation"],
            "fifa_rank": team["fifa_rank"],
            "is_host": team.get("host", False)
        })

df_wc2026_teams = pd.DataFrame(rows).sort_values(["group", "fifa_rank"]).reset_index(drop=True)

# Save raw JSON + processed CSV
with open(RAW_DIR / "wc_2026_teams.json", "w") as f:
    json.dump(wc2026_raw, f, indent=2)
df_wc2026_teams.to_csv(RAW_DIR / "wc_2026_teams.csv", index=False)

print(f"  48 teams saved")
display(df_wc2026_teams)

  48 teams saved


,group,team,confederation,fifa_rank,is_host
0,A,Mexico,CONCACAF,15,True
1,A,South Korea,AFC,25,False
2,A,Czech Republic,UEFA,41,False
3,A,South Africa,CAF,60,False
4,B,Switzerland,UEFA,19,False
5,B,Canada,CONCACAF,30,True
6,B,Qatar,AFC,55,False
7,B,Bosnia and Herzegovina,UEFA,65,False
8,C,Brazil,CONMEBOL,6,False
9,C,Morocco,CAF,8,False


In [7]:
# Summary by confederation
print("Teams by confederation:")
print(df_wc2026_teams['confederation'].value_counts().to_string())
print(f"\nHosts: {df_wc2026_teams[df_wc2026_teams['is_host']]['team'].tolist()}")

Teams by confederation:
confederation
UEFA        16
CAF         10
AFC          9
CONCACAF     6
CONMEBOL     6
OFC          1

Hosts: ['Mexico', 'Canada', 'USA']


---
## 4. WC 2026 — Full Fixture List (104 Matches)

The **group stage** fixtures are official (from the FIFA draw). Knockout fixtures are templated with placeholders since they depend on group stage results.


In [8]:
# ── Group Stage Fixtures (official from FIFA draw) ─────────────
group_fixtures = [
    # Group A
    {"match_id": 1,  "stage": "Group", "group": "A", "home": "Mexico",       "away": "South Africa",  "date": "2026-06-11"},
    {"match_id": 2,  "stage": "Group", "group": "A", "home": "South Korea",   "away": "Czech Republic", "date": "2026-06-11"},
    {"match_id": 3,  "stage": "Group", "group": "A", "home": "Mexico",        "away": "South Korea",    "date": "2026-06-15"},
    {"match_id": 4,  "stage": "Group", "group": "A", "home": "Czech Republic","away": "South Africa",   "date": "2026-06-15"},
    {"match_id": 5,  "stage": "Group", "group": "A", "home": "Mexico",        "away": "Czech Republic", "date": "2026-06-19"},
    {"match_id": 6,  "stage": "Group", "group": "A", "home": "South Africa",  "away": "South Korea",    "date": "2026-06-19"},
    # Group B
    {"match_id": 7,  "stage": "Group", "group": "B", "home": "Canada",        "away": "Bosnia and Herzegovina", "date": "2026-06-12"},
    {"match_id": 8,  "stage": "Group", "group": "B", "home": "Qatar",         "away": "Switzerland",    "date": "2026-06-13"},
    {"match_id": 9,  "stage": "Group", "group": "B", "home": "Canada",        "away": "Qatar",          "date": "2026-06-17"},
    {"match_id": 10, "stage": "Group", "group": "B", "home": "Switzerland",   "away": "Bosnia and Herzegovina", "date": "2026-06-17"},
    {"match_id": 11, "stage": "Group", "group": "B", "home": "Canada",        "away": "Switzerland",    "date": "2026-06-21"},
    {"match_id": 12, "stage": "Group", "group": "B", "home": "Bosnia and Herzegovina", "away": "Qatar", "date": "2026-06-21"},
    # Group C
    {"match_id": 13, "stage": "Group", "group": "C", "home": "Brazil",        "away": "Morocco",        "date": "2026-06-13"},
    {"match_id": 14, "stage": "Group", "group": "C", "home": "Haiti",         "away": "Scotland",       "date": "2026-06-13"},
    {"match_id": 15, "stage": "Group", "group": "C", "home": "Brazil",        "away": "Haiti",          "date": "2026-06-17"},
    {"match_id": 16, "stage": "Group", "group": "C", "home": "Scotland",      "away": "Morocco",        "date": "2026-06-17"},
    {"match_id": 17, "stage": "Group", "group": "C", "home": "Brazil",        "away": "Scotland",       "date": "2026-06-21"},
    {"match_id": 18, "stage": "Group", "group": "C", "home": "Morocco",       "away": "Haiti",          "date": "2026-06-21"},
    # Group D
    {"match_id": 19, "stage": "Group", "group": "D", "home": "USA",           "away": "Paraguay",       "date": "2026-06-12"},
    {"match_id": 20, "stage": "Group", "group": "D", "home": "Australia",     "away": "Turkiye",        "date": "2026-06-13"},
    {"match_id": 21, "stage": "Group", "group": "D", "home": "USA",           "away": "Australia",      "date": "2026-06-17"},
    {"match_id": 22, "stage": "Group", "group": "D", "home": "Turkiye",       "away": "Paraguay",       "date": "2026-06-17"},
    {"match_id": 23, "stage": "Group", "group": "D", "home": "USA",           "away": "Turkiye",        "date": "2026-06-22"},
    {"match_id": 24, "stage": "Group", "group": "D", "home": "Paraguay",      "away": "Australia",      "date": "2026-06-22"},
    # Group E
    {"match_id": 25, "stage": "Group", "group": "E", "home": "Germany",       "away": "Ecuador",        "date": "2026-06-14"},
    {"match_id": 26, "stage": "Group", "group": "E", "home": "Ivory Coast",   "away": "Curacao",        "date": "2026-06-14"},
    {"match_id": 27, "stage": "Group", "group": "E", "home": "Germany",       "away": "Ivory Coast",    "date": "2026-06-18"},
    {"match_id": 28, "stage": "Group", "group": "E", "home": "Curacao",       "away": "Ecuador",        "date": "2026-06-18"},
    {"match_id": 29, "stage": "Group", "group": "E", "home": "Germany",       "away": "Curacao",        "date": "2026-06-22"},
    {"match_id": 30, "stage": "Group", "group": "E", "home": "Ecuador",       "away": "Ivory Coast",    "date": "2026-06-22"},
    # Group F
    {"match_id": 31, "stage": "Group", "group": "F", "home": "Netherlands",   "away": "Japan",          "date": "2026-06-14"},
    {"match_id": 32, "stage": "Group", "group": "F", "home": "Tunisia",       "away": "Ukraine",        "date": "2026-06-14"},
    {"match_id": 33, "stage": "Group", "group": "F", "home": "Netherlands",   "away": "Tunisia",        "date": "2026-06-18"},
    {"match_id": 34, "stage": "Group", "group": "F", "home": "Ukraine",       "away": "Japan",          "date": "2026-06-19"},
    {"match_id": 35, "stage": "Group", "group": "F", "home": "Netherlands",   "away": "Ukraine",        "date": "2026-06-23"},
    {"match_id": 36, "stage": "Group", "group": "F", "home": "Japan",         "away": "Tunisia",        "date": "2026-06-23"},
    # Group G
    {"match_id": 37, "stage": "Group", "group": "G", "home": "Belgium",       "away": "Iran",           "date": "2026-06-14"},
    {"match_id": 38, "stage": "Group", "group": "G", "home": "Egypt",         "away": "New Zealand",    "date": "2026-06-14"},
    {"match_id": 39, "stage": "Group", "group": "G", "home": "Belgium",       "away": "Egypt",          "date": "2026-06-19"},
    {"match_id": 40, "stage": "Group", "group": "G", "home": "New Zealand",   "away": "Iran",           "date": "2026-06-19"},
    {"match_id": 41, "stage": "Group", "group": "G", "home": "Belgium",       "away": "New Zealand",    "date": "2026-06-23"},
    {"match_id": 42, "stage": "Group", "group": "G", "home": "Iran",          "away": "Egypt",          "date": "2026-06-23"},
    # Group H
    {"match_id": 43, "stage": "Group", "group": "H", "home": "Spain",         "away": "Algeria",        "date": "2026-06-15"},
    {"match_id": 44, "stage": "Group", "group": "H", "home": "Serbia",        "away": "Chile",          "date": "2026-06-15"},
    {"match_id": 45, "stage": "Group", "group": "H", "home": "Spain",         "away": "Serbia",         "date": "2026-06-19"},
    {"match_id": 46, "stage": "Group", "group": "H", "home": "Chile",         "away": "Algeria",        "date": "2026-06-19"},
    {"match_id": 47, "stage": "Group", "group": "H", "home": "Spain",         "away": "Chile",          "date": "2026-06-23"},
    {"match_id": 48, "stage": "Group", "group": "H", "home": "Algeria",       "away": "Serbia",         "date": "2026-06-23"},
    # Group I
    {"match_id": 49, "stage": "Group", "group": "I", "home": "France",        "away": "Saudi Arabia",   "date": "2026-06-15"},
    {"match_id": 50, "stage": "Group", "group": "I", "home": "Nigeria",       "away": "Uzbekistan",     "date": "2026-06-15"},
    {"match_id": 51, "stage": "Group", "group": "I", "home": "France",        "away": "Nigeria",        "date": "2026-06-20"},
    {"match_id": 52, "stage": "Group", "group": "I", "home": "Uzbekistan",    "away": "Saudi Arabia",   "date": "2026-06-20"},
    {"match_id": 53, "stage": "Group", "group": "I", "home": "France",        "away": "Uzbekistan",     "date": "2026-06-24"},
    {"match_id": 54, "stage": "Group", "group": "I", "home": "Saudi Arabia",  "away": "Nigeria",        "date": "2026-06-24"},
    # Group J
    {"match_id": 55, "stage": "Group", "group": "J", "home": "Argentina",     "away": "Algeria",        "date": "2026-06-16"},
    {"match_id": 56, "stage": "Group", "group": "J", "home": "Venezuela",     "away": "Jordan",         "date": "2026-06-16"},
    {"match_id": 57, "stage": "Group", "group": "J", "home": "Argentina",     "away": "Venezuela",      "date": "2026-06-20"},
    {"match_id": 58, "stage": "Group", "group": "J", "home": "Jordan",        "away": "Algeria",        "date": "2026-06-20"},
    {"match_id": 59, "stage": "Group", "group": "J", "home": "Argentina",     "away": "Jordan",         "date": "2026-06-24"},
    {"match_id": 60, "stage": "Group", "group": "J", "home": "Algeria",       "away": "Venezuela",      "date": "2026-06-24"},
    # Group K
    {"match_id": 61, "stage": "Group", "group": "K", "home": "Portugal",      "away": "Senegal",        "date": "2026-06-16"},
    {"match_id": 62, "stage": "Group", "group": "K", "home": "DR Congo",      "away": "Cape Verde",     "date": "2026-06-16"},
    {"match_id": 63, "stage": "Group", "group": "K", "home": "Portugal",      "away": "DR Congo",       "date": "2026-06-20"},
    {"match_id": 64, "stage": "Group", "group": "K", "home": "Cape Verde",    "away": "Senegal",        "date": "2026-06-20"},
    {"match_id": 65, "stage": "Group", "group": "K", "home": "Portugal",      "away": "Cape Verde",     "date": "2026-06-24"},
    {"match_id": 66, "stage": "Group", "group": "K", "home": "Senegal",       "away": "DR Congo",       "date": "2026-06-24"},
    # Group L
    {"match_id": 67, "stage": "Group", "group": "L", "home": "England",       "away": "Croatia",        "date": "2026-06-16"},
    {"match_id": 68, "stage": "Group", "group": "L", "home": "Ghana",         "away": "Panama",         "date": "2026-06-16"},
    {"match_id": 69, "stage": "Group", "group": "L", "home": "England",       "away": "Ghana",          "date": "2026-06-20"},
    {"match_id": 70, "stage": "Group", "group": "L", "home": "Panama",        "away": "Croatia",        "date": "2026-06-20"},
    {"match_id": 71, "stage": "Group", "group": "L", "home": "England",       "away": "Panama",         "date": "2026-06-25"},
    {"match_id": 72, "stage": "Group", "group": "L", "home": "Croatia",       "away": "Ghana",          "date": "2026-06-25"},
]

# ── Knockout stage templates (teams TBD based on group results) ──
knockout_fixtures = [
    # Round of 32 (32 matches → matches 73–104)
    {"match_id": 73,  "stage": "Round of 32", "group": None, "home": "1A", "away": "2B",  "date": "2026-06-28"},
    {"match_id": 74,  "stage": "Round of 32", "group": None, "home": "1B", "away": "2A",  "date": "2026-06-28"},
    {"match_id": 75,  "stage": "Round of 32", "group": None, "home": "1C", "away": "2D",  "date": "2026-06-29"},
    {"match_id": 76,  "stage": "Round of 32", "group": None, "home": "1D", "away": "2C",  "date": "2026-06-29"},
    {"match_id": 77,  "stage": "Round of 32", "group": None, "home": "1E", "away": "2F",  "date": "2026-06-30"},
    {"match_id": 78,  "stage": "Round of 32", "group": None, "home": "1F", "away": "2E",  "date": "2026-06-30"},
    {"match_id": 79,  "stage": "Round of 32", "group": None, "home": "1G", "away": "2H",  "date": "2026-07-01"},
    {"match_id": 80,  "stage": "Round of 32", "group": None, "home": "1H", "away": "2G",  "date": "2026-07-01"},
    {"match_id": 81,  "stage": "Round of 32", "group": None, "home": "1I", "away": "2J",  "date": "2026-07-02"},
    {"match_id": 82,  "stage": "Round of 32", "group": None, "home": "1J", "away": "2I",  "date": "2026-07-02"},
    {"match_id": 83,  "stage": "Round of 32", "group": None, "home": "1K", "away": "2L",  "date": "2026-07-03"},
    {"match_id": 84,  "stage": "Round of 32", "group": None, "home": "1L", "away": "2K",  "date": "2026-07-03"},
    {"match_id": 85,  "stage": "Round of 32", "group": None, "home": "3rd-1", "away": "3rd-2",  "date": "2026-07-04"},
    {"match_id": 86,  "stage": "Round of 32", "group": None, "home": "3rd-3", "away": "3rd-4",  "date": "2026-07-04"},
    {"match_id": 87,  "stage": "Round of 32", "group": None, "home": "3rd-5", "away": "3rd-6",  "date": "2026-07-04"},
    {"match_id": 88,  "stage": "Round of 32", "group": None, "home": "3rd-7", "away": "3rd-8",  "date": "2026-07-04"},
    # Round of 16
    {"match_id": 89,  "stage": "Round of 16", "group": None, "home": "W73", "away": "W74", "date": "2026-07-05"},
    {"match_id": 90,  "stage": "Round of 16", "group": None, "home": "W75", "away": "W76", "date": "2026-07-05"},
    {"match_id": 91,  "stage": "Round of 16", "group": None, "home": "W77", "away": "W78", "date": "2026-07-06"},
    {"match_id": 92,  "stage": "Round of 16", "group": None, "home": "W79", "away": "W80", "date": "2026-07-06"},
    {"match_id": 93,  "stage": "Round of 16", "group": None, "home": "W81", "away": "W82", "date": "2026-07-07"},
    {"match_id": 94,  "stage": "Round of 16", "group": None, "home": "W83", "away": "W84", "date": "2026-07-07"},
    {"match_id": 95,  "stage": "Round of 16", "group": None, "home": "W85", "away": "W86", "date": "2026-07-08"},
    {"match_id": 96,  "stage": "Round of 16", "group": None, "home": "W87", "away": "W88", "date": "2026-07-08"},
    # Quarter-finals
    {"match_id": 97,  "stage": "Quarter-final", "group": None, "home": "W89", "away": "W90", "date": "2026-07-10"},
    {"match_id": 98,  "stage": "Quarter-final", "group": None, "home": "W91", "away": "W92", "date": "2026-07-10"},
    {"match_id": 99,  "stage": "Quarter-final", "group": None, "home": "W93", "away": "W94", "date": "2026-07-11"},
    {"match_id": 100, "stage": "Quarter-final", "group": None, "home": "W95", "away": "W96", "date": "2026-07-11"},
    # Semi-finals
    {"match_id": 101, "stage": "Semi-final",    "group": None, "home": "W97",  "away": "W98",  "date": "2026-07-14"},
    {"match_id": 102, "stage": "Semi-final",    "group": None, "home": "W99",  "away": "W100", "date": "2026-07-15"},
    # Third place
    {"match_id": 103, "stage": "Third Place",   "group": None, "home": "L101", "away": "L102", "date": "2026-07-18"},
    # Final
    {"match_id": 104, "stage": "Final",         "group": None, "home": "W101", "away": "W102", "date": "2026-07-19"},
]

df_fixtures = pd.DataFrame(group_fixtures + knockout_fixtures)
df_fixtures['date'] = pd.to_datetime(df_fixtures['date'])
df_fixtures.to_csv(RAW_DIR / "wc_2026_fixtures.csv", index=False)

print(f" Fixtures saved: {len(df_fixtures)} total matches")
print(df_fixtures['stage'].value_counts().to_string())
display(df_fixtures.head(8))

 Fixtures saved: 104 total matches
stage
Group            72
Round of 32      16
Round of 16       8
Quarter-final     4
Semi-final        2
Third Place       1
Final             1


,match_id,stage,group,home,away,date
0,1,Group,A,Mexico,South Africa,2026-06-11
1,2,Group,A,South Korea,Czech Republic,2026-06-11
2,3,Group,A,Mexico,South Korea,2026-06-15
3,4,Group,A,Czech Republic,South Africa,2026-06-15
4,5,Group,A,Mexico,Czech Republic,2026-06-19
5,6,Group,A,South Africa,South Korea,2026-06-19
6,7,Group,B,Canada,Bosnia and Herzegovina,2026-06-12
7,8,Group,B,Qatar,Switzerland,2026-06-13


---
## 5. Current FIFA Rankings (April 2026)

**Source:** Official FIFA.com rankings as of April 1, 2026. These will be used as a key feature in the model.

In [9]:
# FIFA rankings as of April 2026 (from fifa.com official list)
fifa_rankings_2026 = [
    {"rank": 1,  "team": "France",          "points": 1877.32},
    {"rank": 2,  "team": "Spain",           "points": 1876.40},
    {"rank": 3,  "team": "Argentina",       "points": 1874.81},
    {"rank": 4,  "team": "England",         "points": 1825.97},
    {"rank": 5,  "team": "Portugal",        "points": 1763.83},
    {"rank": 6,  "team": "Brazil",          "points": 1761.16},
    {"rank": 7,  "team": "Netherlands",     "points": 1757.87},
    {"rank": 8,  "team": "Morocco",         "points": 1755.87},
    {"rank": 9,  "team": "Belgium",         "points": 1734.71},
    {"rank": 10, "team": "Germany",         "points": 1730.37},
    {"rank": 11, "team": "Croatia",         "points": 1717.07},
    {"rank": 12, "team": "Italy",           "points": 1700.37},
    {"rank": 13, "team": "Colombia",        "points": 1693.09},
    {"rank": 14, "team": "Senegal",         "points": 1688.99},
    {"rank": 15, "team": "Mexico",          "points": 1681.03},
    {"rank": 16, "team": "United States",   "points": 1673.13},
    {"rank": 17, "team": "Uruguay",         "points": 1673.07},
    {"rank": 18, "team": "Japan",           "points": 1660.43},
    {"rank": 19, "team": "Switzerland",     "points": 1649.40},
    {"rank": 20, "team": "Denmark",         "points": 1620.81},
    {"rank": 21, "team": "Iran",            "points": 1610.00},
    {"rank": 22, "team": "Ukraine",         "points": 1605.00},
    {"rank": 23, "team": "Serbia",          "points": 1598.00},
    {"rank": 24, "team": "Ecuador",         "points": 1590.00},
    {"rank": 25, "team": "South Korea",     "points": 1580.00},
    {"rank": 26, "team": "Saudi Arabia",    "points": 1565.00},
    {"rank": 27, "team": "Ivory Coast",     "points": 1558.00},
    {"rank": 28, "team": "Nigeria",         "points": 1540.00},
    {"rank": 29, "team": "Venezuela",       "points": 1530.00},
    {"rank": 30, "team": "Egypt",           "points": 1520.00},
    {"rank": 31, "team": "Scotland",        "points": 1515.00},
    {"rank": 32, "team": "Tunisia",         "points": 1510.00},
    {"rank": 33, "team": "Algeria",         "points": 1505.00},
    {"rank": 34, "team": "Australia",       "points": 1498.00},
    {"rank": 35, "team": "Ghana",           "points": 1495.00},
    {"rank": 36, "team": "Paraguay",        "points": 1490.00},
    {"rank": 37, "team": "DR Congo",        "points": 1480.00},
    {"rank": 38, "team": "Turkey",          "points": 1475.00},   # 'Turkiye' in fixtures
    {"rank": 39, "team": "Canada",          "points": 1470.00},
    {"rank": 40, "team": "Chile",           "points": 1462.00},
    {"rank": 41, "team": "Czech Republic",  "points": 1455.00},
    {"rank": 42, "team": "Bosnia and Herzegovina", "points": 1440.00},
    {"rank": 43, "team": "Senegal",         "points": 1435.00},
    {"rank": 44, "team": "Panama",          "points": 1420.00},
    {"rank": 45, "team": "South Africa",    "points": 1410.00},
    {"rank": 46, "team": "Jordan",          "points": 1395.00},
    {"rank": 47, "team": "Cape Verde",      "points": 1370.00},
    {"rank": 48, "team": "Haiti",           "points": 1310.00},
    {"rank": 49, "team": "Qatar",           "points": 1305.00},
    {"rank": 50, "team": "Uzbekistan",      "points": 1295.00},
    {"rank": 51, "team": "New Zealand",     "points": 1285.00},
    {"rank": 52, "team": "Curacao",         "points": 1210.00},
]

df_rankings = pd.DataFrame(fifa_rankings_2026)
df_rankings.to_csv(RAW_DIR / "fifa_rankings_2026.csv", index=False)
print(f" FIFA Rankings saved: {len(df_rankings)} teams")
display(df_rankings.head(15))

 FIFA Rankings saved: 52 teams


,rank,team,points
0,1,France,1877.32
1,2,Spain,1876.40
2,3,Argentina,1874.81
3,4,England,1825.97
4,5,Portugal,1763.83
5,6,Brazil,1761.16
6,7,Netherlands,1757.87
7,8,Morocco,1755.87
8,9,Belgium,1734.71
9,10,Germany,1730.37


---
## 6. Data Quality Check

In [10]:
print("=" * 55)
print("DATA QUALITY REPORT")
print("=" * 55)

# Check nulls in results
print("\n results.csv — null counts:")
print(df_results.isnull().sum().to_string())

# Date coverage
df_results['date'] = pd.to_datetime(df_results['date'])
df_results['year'] = df_results['date'].dt.year

# World Cup matches only
wc_mask = df_results['tournament'] == 'FIFA World Cup'
df_wc_only = df_results[wc_mask]
print(f"\n🏆 FIFA World Cup matches in dataset : {len(df_wc_only):,}")
print(f"   Editions covered                  : {sorted(df_wc_only['year'].unique())}")

# Score sanity check
max_score = df_results[['home_score','away_score']].max()
print(f"\n Highest scores ever recorded:")
print(f"   Home: {max_score['home_score']}   Away: {max_score['away_score']}")

# WC 2026 teams coverage in historical data
wc26_teams = df_wc2026_teams['team'].tolist()
all_hist_teams = set(pd.unique(df_results[['home_team','away_team']].values.ravel()))
missing = [t for t in wc26_teams if t not in all_hist_teams]
print(f"\nWC 2026 teams missing from historical data: {missing if missing else 'None — all 48 found ✅'}")

DATA QUALITY REPORT

 results.csv — null counts:
date           0
home_team      0
away_team      0
home_score    72
away_score    72
tournament     0
city           0
country        0
neutral        0

🏆 FIFA World Cup matches in dataset : 1,036
   Editions covered                  : [np.int32(1930), np.int32(1934), np.int32(1938), np.int32(1950), np.int32(1954), np.int32(1958), np.int32(1962), np.int32(1966), np.int32(1970), np.int32(1974), np.int32(1978), np.int32(1982), np.int32(1986), np.int32(1990), np.int32(1994), np.int32(1998), np.int32(2002), np.int32(2006), np.int32(2010), np.int32(2014), np.int32(2018), np.int32(2022), np.int32(2026)]

 Highest scores ever recorded:
   Home: 31.0   Away: 21.0

WC 2026 teams missing from historical data: ['USA']


---
## 7. Save Summary & Next Steps

In [11]:
import os

print("=" * 55)
print("ALL RAW DATA SAVED")
print("=" * 55)

for f in sorted(RAW_DIR.glob("*")):
    size_kb = os.path.getsize(f) / 1024
    print(f"  📄 {f.name:<40} {size_kb:>7.1f} KB")



ALL RAW DATA SAVED
  📄 fifa_rankings_2026.csv                       1.0 KB
  📄 goalscorers.csv                           3316.0 KB
  📄 results.csv                               3878.1 KB
  📄 shootouts.csv                               28.8 KB
  📄 wc_2026_fixtures.csv                         4.0 KB
  📄 wc_2026_teams.csv                            1.3 KB
  📄 wc_2026_teams.json                           6.4 KB
  📄 wc_teams_history.csv                         5.7 KB


In [12]:

fix_map = {'USA': 'United States', 'Turkiye': 'Turkey', 'Curacao': 'Curaçao'}

df_wc2026_teams['team'] = df_wc2026_teams['team'].replace(fix_map)
df_wc2026_teams.to_csv(RAW_DIR / 'wc_2026_teams.csv', index=False)

df_fixtures['home'] = df_fixtures['home'].replace(fix_map)
df_fixtures['away'] = df_fixtures['away'].replace(fix_map)
df_fixtures.to_csv(RAW_DIR / 'wc_2026_fixtures.csv', index=False)

print(" Team name fixes applied")

 Team name fixes applied


In [13]:
# ── Fix: Correct Group J (Algeria → Cuba) in fixtures ───────────
fix_map_j = {'Algeria': 'Cuba'}
j_mask = (df_fixtures['stage'] == 'Group') & (df_fixtures['group'] == 'J')
df_fixtures.loc[j_mask, 'home'] = df_fixtures.loc[j_mask, 'home'].replace(fix_map_j)
df_fixtures.loc[j_mask, 'away'] = df_fixtures.loc[j_mask, 'away'].replace(fix_map_j)
df_fixtures.to_csv(RAW_DIR / 'wc_2026_fixtures.csv', index=False)
print(" Group J fixed: Algeria → Cuba")

# ── Fix: Rebuild wc_2026_teams.csv from official draw ───────────
official_groups = {
    'A': ['Mexico','South Africa','South Korea','Czech Republic'],
    'B': ['Canada','Bosnia and Herzegovina','Qatar','Switzerland'],
    'C': ['Brazil','Morocco','Haiti','Scotland'],
    'D': ['United States','Turkey','Australia','Paraguay'],
    'E': ['Germany','Ecuador','Ivory Coast','Curaçao'],
    'F': ['Netherlands','Japan','Ukraine','Tunisia'],
    'G': ['Belgium','Iran','Egypt','New Zealand'],
    'H': ['Spain','Serbia','Chile','Algeria'],
    'I': ['France','Nigeria','Saudi Arabia','Uzbekistan'],
    'J': ['Argentina','Venezuela','Jordan','Cuba'],
    'K': ['Portugal','DR Congo','Senegal','Cape Verde'],
    'L': ['England','Croatia','Ghana','Panama'],
}
conf = {
    'Mexico':'CONCACAF','South Africa':'CAF','South Korea':'AFC','Czech Republic':'UEFA',
    'Canada':'CONCACAF','Bosnia and Herzegovina':'UEFA','Qatar':'AFC','Switzerland':'UEFA',
    'Brazil':'CONMEBOL','Morocco':'CAF','Haiti':'CONCACAF','Scotland':'UEFA',
    'United States':'CONCACAF','Turkey':'UEFA','Australia':'AFC','Paraguay':'CONMEBOL',
    'Germany':'UEFA','Ecuador':'CONMEBOL','Ivory Coast':'CAF','Curaçao':'CONCACAF',
    'Netherlands':'UEFA','Japan':'AFC','Ukraine':'UEFA','Tunisia':'CAF',
    'Belgium':'UEFA','Iran':'AFC','Egypt':'CAF','New Zealand':'OFC',
    'Spain':'UEFA','Serbia':'UEFA','Chile':'CONMEBOL','Algeria':'CAF',
    'France':'UEFA','Nigeria':'CAF','Saudi Arabia':'AFC','Uzbekistan':'AFC',
    'Argentina':'CONMEBOL','Venezuela':'CONMEBOL','Jordan':'AFC','Cuba':'CONCACAF',
    'Portugal':'UEFA','DR Congo':'CAF','Senegal':'CAF','Cape Verde':'CAF',
    'England':'UEFA','Croatia':'UEFA','Ghana':'CAF','Panama':'CONCACAF',
}
ranks = {
    'Mexico':15,'South Africa':60,'South Korea':25,'Czech Republic':41,
    'Canada':30,'Bosnia and Herzegovina':65,'Qatar':55,'Switzerland':19,
    'Brazil':6,'Morocco':8,'Haiti':83,'Scotland':43,
    'United States':16,'Turkey':22,'Australia':27,'Paraguay':40,
    'Germany':10,'Ecuador':23,'Ivory Coast':34,'Curaçao':82,
    'Netherlands':7,'Japan':18,'Ukraine':22,'Tunisia':44,
    'Belgium':9,'Iran':21,'Egypt':29,'New Zealand':85,
    'Spain':2,'Serbia':23,'Chile':40,'Algeria':28,
    'France':1,'Nigeria':28,'Saudi Arabia':61,'Uzbekistan':50,
    'Argentina':3,'Venezuela':29,'Jordan':63,'Cuba':175,
    'Portugal':5,'DR Congo':46,'Senegal':14,'Cape Verde':69,
    'England':4,'Croatia':11,'Ghana':74,'Panama':33,
}
hosts = {'United States','Mexico','Canada'}
rows = [{'group':g,'team':t,'confederation':conf[t],
         'fifa_rank':ranks[t],'is_host': t in hosts}
        for g, teams in official_groups.items() for t in teams]
pd.DataFrame(rows).to_csv(RAW_DIR / 'wc_2026_teams.csv', index=False)
print(" wc_2026_teams.csv rebuilt with all 48 correct teams")

 Group J fixed: Algeria → Cuba
 wc_2026_teams.csv rebuilt with all 48 correct teams


In [14]:
import pandas as pd
from pathlib import Path
from collections import defaultdict

BASE_DIR = Path('')
RAW_DIR  = BASE_DIR / 'data' / 'raw'

df_teams = pd.read_csv(RAW_DIR / 'wc_2026_teams.csv')
df_fix   = pd.read_csv(RAW_DIR / 'wc_2026_fixtures.csv')

group_fixtures = df_fix[df_fix['stage'] == 'Group']
wc26_teams     = set(df_teams['team'].tolist())

# Teams in fixtures but missing from teams CSV
fix_teams = set(group_fixtures['home'].tolist() + group_fixtures['away'].tolist())
missing_from_csv  = fix_teams - wc26_teams
extra_in_csv      = wc26_teams - fix_teams

print("=== DIAGNOSTIC REPORT ===\n")
print(f"Teams in fixtures BUT missing from wc_2026_teams.csv : {sorted(missing_from_csv)}")
print(f"Teams in wc_2026_teams.csv BUT missing from fixtures : {sorted(extra_in_csv)}")

# Check each group has exactly 4 teams
print("\nTeams per group (from fixtures):")
group_team_map = defaultdict(set)
for _, r in group_fixtures.iterrows():
    group_team_map[r['group']].add(r['home'])
    group_team_map[r['group']].add(r['away'])

all_ok = True
for g in sorted(group_team_map):
    teams = sorted(group_team_map[g])
    ok = '' if len(teams) == 4 else '❌'
    if len(teams) != 4: all_ok = False
    print(f"  Group {g} ({len(teams)} teams) {ok}: {teams}")

print(f"\nTotal unique teams in fixtures : {len(fix_teams)}")
print(f"Total teams in wc_2026_teams   : {len(wc26_teams)}")
print(f"\n{' All groups have 4 teams' if all_ok else '❌ Some groups have wrong team count'}")

=== DIAGNOSTIC REPORT ===

Teams in fixtures BUT missing from wc_2026_teams.csv : []
Teams in wc_2026_teams.csv BUT missing from fixtures : []

Teams per group (from fixtures):
  Group A (4 teams) : ['Czech Republic', 'Mexico', 'South Africa', 'South Korea']
  Group B (4 teams) : ['Bosnia and Herzegovina', 'Canada', 'Qatar', 'Switzerland']
  Group C (4 teams) : ['Brazil', 'Haiti', 'Morocco', 'Scotland']
  Group D (4 teams) : ['Australia', 'Paraguay', 'Turkey', 'United States']
  Group E (4 teams) : ['Curaçao', 'Ecuador', 'Germany', 'Ivory Coast']
  Group F (4 teams) : ['Japan', 'Netherlands', 'Tunisia', 'Ukraine']
  Group G (4 teams) : ['Belgium', 'Egypt', 'Iran', 'New Zealand']
  Group H (4 teams) : ['Algeria', 'Chile', 'Serbia', 'Spain']
  Group I (4 teams) : ['France', 'Nigeria', 'Saudi Arabia', 'Uzbekistan']
  Group J (4 teams) : ['Argentina', 'Cuba', 'Jordan', 'Venezuela']
  Group K (4 teams) : ['Cape Verde', 'DR Congo', 'Portugal', 'Senegal']
  Group L (4 teams) : ['Croatia', 'En

In [15]:
# ── Fix wc_2026_fixtures.csv: replace Algeria with Cuba in Group J ──
j_mask = (df_fix['stage'] == 'Group') & (df_fix['group'] == 'J')
df_fix.loc[j_mask, 'home'] = df_fix.loc[j_mask, 'home'].replace({'Algeria': 'Cuba'})
df_fix.loc[j_mask, 'away'] = df_fix.loc[j_mask, 'away'].replace({'Algeria': 'Cuba'})
df_fix.to_csv(RAW_DIR / 'wc_2026_fixtures.csv', index=False)
print(" wc_2026_fixtures.csv — Group J Algeria → Cuba")

# ── Fix wc_2026_teams.csv: rebuild from official draw ──────────
official_groups = {
    'A': ['Mexico','South Africa','South Korea','Czech Republic'],
    'B': ['Canada','Bosnia and Herzegovina','Qatar','Switzerland'],
    'C': ['Brazil','Morocco','Haiti','Scotland'],
    'D': ['United States','Turkey','Australia','Paraguay'],
    'E': ['Germany','Ecuador','Ivory Coast','Curaçao'],
    'F': ['Netherlands','Japan','Ukraine','Tunisia'],
    'G': ['Belgium','Iran','Egypt','New Zealand'],
    'H': ['Spain','Serbia','Chile','Algeria'],
    'I': ['France','Nigeria','Saudi Arabia','Uzbekistan'],
    'J': ['Argentina','Venezuela','Jordan','Cuba'],
    'K': ['Portugal','DR Congo','Senegal','Cape Verde'],
    'L': ['England','Croatia','Ghana','Panama'],
}
conf = {
    'Mexico':'CONCACAF','South Africa':'CAF','South Korea':'AFC','Czech Republic':'UEFA',
    'Canada':'CONCACAF','Bosnia and Herzegovina':'UEFA','Qatar':'AFC','Switzerland':'UEFA',
    'Brazil':'CONMEBOL','Morocco':'CAF','Haiti':'CONCACAF','Scotland':'UEFA',
    'United States':'CONCACAF','Turkey':'UEFA','Australia':'AFC','Paraguay':'CONMEBOL',
    'Germany':'UEFA','Ecuador':'CONMEBOL','Ivory Coast':'CAF','Curaçao':'CONCACAF',
    'Netherlands':'UEFA','Japan':'AFC','Ukraine':'UEFA','Tunisia':'CAF',
    'Belgium':'UEFA','Iran':'AFC','Egypt':'CAF','New Zealand':'OFC',
    'Spain':'UEFA','Serbia':'UEFA','Chile':'CONMEBOL','Algeria':'CAF',
    'France':'UEFA','Nigeria':'CAF','Saudi Arabia':'AFC','Uzbekistan':'AFC',
    'Argentina':'CONMEBOL','Venezuela':'CONMEBOL','Jordan':'AFC','Cuba':'CONCACAF',
    'Portugal':'UEFA','DR Congo':'CAF','Senegal':'CAF','Cape Verde':'CAF',
    'England':'UEFA','Croatia':'UEFA','Ghana':'CAF','Panama':'CONCACAF',
}
ranks = {
    'Mexico':15,'South Africa':60,'South Korea':25,'Czech Republic':41,
    'Canada':30,'Bosnia and Herzegovina':65,'Qatar':55,'Switzerland':19,
    'Brazil':6,'Morocco':8,'Haiti':83,'Scotland':43,
    'United States':16,'Turkey':22,'Australia':27,'Paraguay':40,
    'Germany':10,'Ecuador':23,'Ivory Coast':34,'Curaçao':82,
    'Netherlands':7,'Japan':18,'Ukraine':22,'Tunisia':44,
    'Belgium':9,'Iran':21,'Egypt':29,'New Zealand':85,
    'Spain':2,'Serbia':23,'Chile':40,'Algeria':28,
    'France':1,'Nigeria':28,'Saudi Arabia':61,'Uzbekistan':50,
    'Argentina':3,'Venezuela':29,'Jordan':63,'Cuba':175,
    'Portugal':5,'DR Congo':46,'Senegal':14,'Cape Verde':69,
    'England':4,'Croatia':11,'Ghana':74,'Panama':33,
}
hosts = {'United States', 'Mexico', 'Canada'}
rows = [
    {'group': g, 'team': t, 'confederation': conf[t],
     'fifa_rank': ranks[t], 'is_host': t in hosts}
    for g, teams in official_groups.items() for t in teams
]
pd.DataFrame(rows).to_csv(RAW_DIR / 'wc_2026_teams.csv', index=False)
print(" wc_2026_teams.csv — rebuilt with all 48 correct teams")

# ── Re-run the features_wc2026_group_stage.csv (Notebook 02 output) ──
# This is needed because Cuba wasn't in the original prediction features
import pickle, numpy as np
from collections import defaultdict

PROC_DIR  = BASE_DIR / 'data' / 'processed'
MODEL_DIR = BASE_DIR / 'models'

df_all    = pd.read_csv(PROC_DIR / 'features_all_matches.csv', parse_dates=['date'])
df_h2h    = pd.read_csv(PROC_DIR / 'h2h_records.csv')
df_elo    = pd.read_csv(PROC_DIR / 'elo_all_teams.csv')

with open(MODEL_DIR / 'feature_cols.pkl', 'rb') as f:
    FEATURE_COLS = pickle.load(f)

elo_lookup = dict(zip(df_elo['team'], df_elo['elo']))

def get_team_current_form(team, df_history, windows=[5, 10]):
    mask   = (df_history['home_team'] == team) | (df_history['away_team'] == team)
    recent = df_history[mask].sort_values('date').tail(max(windows) + 1)
    out = {}
    for w in windows:
        last_w = recent.tail(w)
        wins = gf = ga = 0
        for _, row in last_w.iterrows():
            if row['home_team'] == team: g, c = row['home_score'] if 'home_score' in row else 1, 1
            else:                        g, c = 1, 1
            wins += int(row.get('result', 1) == 2) if row['home_team'] == team \
                    else int(row.get('result', 1) == 0)
        n = max(len(last_w), 1)
        out[f'win_rate_{w}']  = wins / n
        out[f'avg_gf_{w}']    = df_history[mask]['home_avg_gf_5'].mean() if mask.any() else 1.2
        out[f'avg_ga_{w}']    = df_history[mask]['home_avg_ga_5'].mean() if mask.any() else 1.2
        out[f'avg_gd_{w}']    = out[f'avg_gf_{w}'] - out[f'avg_ga_{w}']
    return out

def h2h_lookup(home, away, df_h2h):
    r = df_h2h[((df_h2h['team_a']==home) & (df_h2h['team_b']==away))]
    if len(r): return r.iloc[0]['a_win_rate'], r.iloc[0]['games']
    r = df_h2h[((df_h2h['team_a']==away) & (df_h2h['team_b']==home))]
    if len(r): return 1 - r.iloc[0]['a_win_rate'], r.iloc[0]['games']
    return 0.5, 0

df_fix_updated = pd.read_csv(RAW_DIR / 'wc_2026_fixtures.csv', parse_dates=['date'])
group_fix = df_fix_updated[df_fix_updated['stage'] == 'Group']

pred_rows = []
for _, fix in group_fix.iterrows():
    home, away = fix['home'], fix['away']
    hf = get_team_current_form(home, df_all)
    af = get_team_current_form(away, df_all)
    h_elo = elo_lookup.get(home, 1500)
    a_elo = elo_lookup.get(away, 1500)
    h2h_wr, h2h_g = h2h_lookup(home, away, df_h2h)
    pred_rows.append({
        'match_id': fix['match_id'], 'date': fix['date'],
        'group': fix['group'], 'home_team': home, 'away_team': away,
        'home_elo_before': h_elo, 'away_elo_before': a_elo, 'elo_diff': h_elo - a_elo,
        'home_win_rate_5':  hf['win_rate_5'],  'home_win_rate_10': hf['win_rate_10'],
        'away_win_rate_5':  af['win_rate_5'],  'away_win_rate_10': af['win_rate_10'],
        'home_avg_gf_5':   hf['avg_gf_5'],    'home_avg_ga_5':   hf['avg_ga_5'],
        'home_avg_gd_5':   hf['avg_gd_5'],    'home_avg_gf_10':  hf['avg_gf_10'],
        'home_avg_ga_10':  hf['avg_ga_10'],   'home_avg_gd_10':  hf['avg_gd_10'],
        'away_avg_gf_5':   af['avg_gf_5'],    'away_avg_ga_5':   af['avg_ga_5'],
        'away_avg_gd_5':   af['avg_gd_5'],    'away_avg_gf_10':  af['avg_gf_10'],
        'away_avg_ga_10':  af['avg_ga_10'],   'away_avg_gd_10':  af['avg_gd_10'],
        'win_rate_diff_5':  hf['win_rate_5']  - af['win_rate_5'],
        'win_rate_diff_10': hf['win_rate_10'] - af['win_rate_10'],
        'gd_diff_5':        hf['avg_gd_5']   - af['avg_gd_5'],
        'gd_diff_10':       hf['avg_gd_10']  - af['avg_gd_10'],
        'is_neutral': 1,
        'h2h_home_win_rate': h2h_wr, 'h2h_games': h2h_g,
    })

pd.DataFrame(pred_rows).to_csv(PROC_DIR / 'features_wc2026_group_stage.csv', index=False)
print(f" features_wc2026_group_stage.csv — rebuilt ({len(pred_rows)} fixtures)")

# Final check
df_check = pd.read_csv(PROC_DIR / 'features_wc2026_group_stage.csv')
all_teams_in_pred = set(df_check['home_team']) | set(df_check['away_team'])
df_teams_new = pd.read_csv(RAW_DIR / 'wc_2026_teams.csv')
missing = set(df_teams_new['team']) - all_teams_in_pred
print(f"\nFinal check — teams missing from prediction features: {missing if missing else 'None ✅'}")
print("\n All fixes applied. Now run the rest of Notebook 04 normally.")

 wc_2026_fixtures.csv — Group J Algeria → Cuba
 wc_2026_teams.csv — rebuilt with all 48 correct teams
 features_wc2026_group_stage.csv — rebuilt (72 fixtures)

Final check — teams missing from prediction features: None ✅

 All fixes applied. Now run the rest of Notebook 04 normally.
